# Notebook 03 — RAG Pipeline

**Steps:**
1. Load **toàn bộ** chunks từ `all_chunks.jsonl`
2. Encode với `bkai-foundation-models/vietnamese-bi-encoder`
3. Build FAISS `IndexFlatIP` (cosine similarity)
4. Lưu index + chunk metadata
5. Retriever function
6. Sanity check chất lượng retrieval

> **Prerequisite:** Run `01_setup_data_prep.ipynb` trước.
>
> **Lưu ý:** Toàn bộ 20 file (không lọc split) đều được index.  
> Điều này đảm bảo retriever có thể tìm chunk cho mọi câu hỏi trong test set → Recall@5 đo được chính xác.

In [2]:
from pathlib import Path
import os

# Tự động tìm đường dẫn
if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NLP_Final/Source/')
else:
    BASE = "../"

print(f'Base directory: {BASE}')
print(os.listdir(BASE))

Mounted at /content/drive
Base directory: /content/drive/MyDrive/NLP_Final/Source
['.gitignore', '.env', 'requirements.txt', 'scripts', '.git', 'data', 'notebooks', 'models', 'results']


In [ ]:
# Fix NumPy 2.x không tương thích với faiss-cpu được compile cho NumPy 1.x
# Phải downgrade numpy TRƯỚC khi import faiss
!pip install -q "numpy<2" faiss-cpu==1.8.0 sentence-transformers==3.2.1


## 1. Load Toàn Bộ Chunks

In [ ]:
import json, os

# Load child chunks để index vào FAISS
all_chunks = []
with open(f'{BASE}/data/chunks/all_chunks.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            all_chunks.append(json.loads(line))

# Load parent chunks → dùng để inject context đầy đủ vào LLM prompt
parent_map = {}  # parent_chunk_id → parent_text
with open(f'{BASE}/data/chunks/parent_chunks.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            p = json.loads(line)
            parent_map[p['parent_chunk_id']] = p['text']

index_chunks = all_chunks  # embed tất cả child chunks

print(f'Child chunks to index: {len(index_chunks)}')
print(f'Parent chunks loaded : {len(parent_map)}')
print(f'Files covered        : {len(set(c["source_file"] for c in index_chunks))}/20')

Child chunks to index: 850
Parent chunks loaded : 232
Files covered        : 20/20


## 2. Load Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = 'bkai-foundation-models/vietnamese-bi-encoder'
embedder = SentenceTransformer(EMBED_MODEL)
print(f'Embedding model: {EMBED_MODEL}')
print(f'Embedding dim  : {embedder.get_sentence_embedding_dimension()}')

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Embedding model: bkai-foundation-models/vietnamese-bi-encoder
Embedding dim  : 768


## 3. Encode Toàn Bộ Chunks (dùng `embed_text`)

Sau khi agentic chunking, mỗi chunk có trường `embed_text = title + summary + text`.  
Embedding `embed_text` thay vì chỉ `text` → retrieval tốt hơn vì title/summary chứa từ khóa ngữ nghĩa khớp với query của người dùng.

In [ ]:
import numpy as np

# Dùng embed_text (title + summary + text) nếu có, fallback về text
texts = [c.get('embed_text') or c['text'] for c in index_chunks]

has_embed_text = sum(1 for c in index_chunks if c.get('embed_text'))
print(f'Encoding {len(texts)} chunks...')
print(f'  Chunks có embed_text (agentic labeled): {has_embed_text}')
print(f'  Chunks dùng text thuần: {len(texts) - has_embed_text}')

embeddings = embedder.encode(
    texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.array(embeddings, dtype='float32')
print(f'Embeddings shape: {embeddings.shape}')

Encoding 850 chunks...
  Chunks có embed_text (agentic labeled): 850
  Chunks dùng text thuần: 0


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Embeddings shape: (850, 768)


## 4. Build FAISS Index

In [ ]:
import faiss

DIMENSION = embeddings.shape[1]  # 768

# IndexFlatIP: exact inner product search
# Với L2-normalized vectors → tương đương cosine similarity
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)

print(f'FAISS index: {index.ntotal} vectors, dim={DIMENSION}')

FAISS index: 850 vectors, dim=768


## 5. Save Index + Chunk Metadata

In [ ]:
import pickle

INDEX_DIR  = f'{BASE}/data/vector_store/faiss_index'
INDEX_PATH = f'{INDEX_DIR}/index.faiss'
META_PATH  = f'{INDEX_DIR}/index.pkl'

os.makedirs(INDEX_DIR, exist_ok=True)
faiss.write_index(index, INDEX_PATH)

with open(META_PATH, 'wb') as f:
    pickle.dump(index_chunks, f)

print(f'Saved FAISS index    → {INDEX_PATH}')
print(f'Saved chunk metadata → {META_PATH}')

Saved FAISS index    → /content/drive/MyDrive/NLP_Final/Source/data/vector_store/faiss_index/index.faiss
Saved chunk metadata → /content/drive/MyDrive/NLP_Final/Source/data/vector_store/faiss_index/index.pkl


## 6. Retriever Function

In [ ]:
def retrieve(query: str, k: int = 5) -> list[dict]:
    """Return top-k most similar child chunks for a given query."""
    q_vec = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_vec, k)
    return [
        {'chunk': index_chunks[i], 'score': float(scores[0][j])}
        for j, i in enumerate(indices[0]) if i < len(index_chunks)
    ]


def format_context(results: list[dict], top_n: int = 3) -> str:
    """
    Format context cho LLM prompt.

    Chiến lược parent-child:
    - Retrieval dùng child chunk (nhỏ, embedding chính xác)
    - LLM nhận parent_text (đầy đủ ngữ nghĩa, không thiếu điều kiện)
    - Dedup theo parent_chunk_id — tránh inject cùng 1 đoạn nhiều lần
    """
    seen_parents: set = set()
    parts = []
    for r in results:
        chunk = r['chunk']
        pid = chunk.get('parent_chunk_id', '')
        # Lookup parent text từ parent_map
        ctx = parent_map.get(pid) or chunk['text']
        if pid and pid in seen_parents:
            continue
        if pid:
            seen_parents.add(pid)
        parts.append(f"[{len(parts)+1}] {ctx}")
        if len(parts) >= top_n:
            break
    return '\n\n'.join(parts)


print('Retriever + format_context ready.')
print('→ format_context() tra parent_map bằng parent_chunk_id — context đầy đủ.')

Retriever + format_context ready.
→ format_context() tra parent_map bằng parent_chunk_id — context đầy đủ.


## 7. Load Index at Runtime (helper cho các notebook khác)

In [ ]:
def load_retriever(base: str):
    """
    Load FAISS index, chunk metadata, và parent_map.
    Dùng trong NB05 và NB07.

    Returns: (_retrieve_fn, _format_context_fn)
    """
    import faiss, pickle, json
    from sentence_transformers import SentenceTransformer

    index_dir = f'{base}/data/vector_store/faiss_index'
    idx = faiss.read_index(f'{index_dir}/index.faiss')
    with open(f'{index_dir}/index.pkl', 'rb') as f:
        chunks = pickle.load(f)

    # Load parent_map để format_context dùng
    _parent_map = {}
    parents_path = f'{base}/data/chunks/parent_chunks.jsonl'
    if os.path.exists(parents_path):
        with open(parents_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    p = json.loads(line)
                    _parent_map[p['parent_chunk_id']] = p['text']

    emb = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder')

    def _retrieve(query: str, k: int = 5):
        q_vec = emb.encode([query], normalize_embeddings=True).astype('float32')
        scores, indices = idx.search(q_vec, k)
        return [{'chunk': chunks[i], 'score': float(scores[0][j])}
                for j, i in enumerate(indices[0]) if i < len(chunks)]

    def _format_context(results, top_n: int = 3) -> str:
        seen, parts = set(), []
        for r in results:
            pid = r['chunk'].get('parent_chunk_id', '')
            ctx = _parent_map.get(pid) or r['chunk']['text']
            if pid and pid in seen:
                continue
            if pid:
                seen.add(pid)
            parts.append(f"[{len(parts)+1}] {ctx}")
            if len(parts) >= top_n:
                break
        return '\n\n'.join(parts)

    print(f'Retriever loaded: {idx.ntotal} vectors | {len(_parent_map)} parents')
    return _retrieve, _format_context

print('load_retriever() helper defined.')

load_retriever() helper defined.


## 8. Sanity Check — Recall@5

In [ ]:
# Load test QA pairs
test_pairs = []
with open(f'{BASE}/data/qa_filtered/qa_test.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            test_pairs.append(json.loads(line))

# Tạo lookup: chunk_id → parent_chunk_id (để map QA pair về parent)
child_to_parent = {c['chunk_id']: c.get('parent_chunk_id', '') for c in index_chunks}

sanity_set = test_pairs[:10]
hits_at_5  = 0

for pair in sanity_set:
    results = retrieve(pair['question'], k=5)
    # Recall đúng: hit nếu BẤT KỲ child nào của đúng parent được retrieve
    # (vì QA sinh từ parent, không phải từ 1 child cụ thể)
    gold_parent = pair.get('parent_chunk_id') or child_to_parent.get(pair.get('chunk_id', ''), '')
    retrieved_parents = [child_to_parent.get(r['chunk']['chunk_id'], '') for r in results]
    hit = bool(gold_parent and gold_parent in retrieved_parents)
    hits_at_5 += int(hit)
    status = 'OKE' if hit else 'KO'
    print(f'{status} Q: {pair["question"][:70]}')
    if not hit:
        print(f'   Gold parent: {gold_parent}')
        print(f'   Retrieved  : {retrieved_parents}')

recall = hits_at_5 / len(sanity_set) if sanity_set else 0
print(f'\nSanity Recall@5 (parent-level): {hits_at_5}/{len(sanity_set)} = {recall:.0%}')
if recall < 0.8:
    print('Dưới 80%. Xem xét điều chỉnh embedding model hoặc chunk size.')
else:
    print('Retrieval chất lượng tốt.')

OKE Q: Khi em nghe điện thoại từ thầy cô gọi đến, em cần trả lời trong bao lâ
OKE Q: Em và bạn cùng lớp xảy ra mâu thuẫn, em nên giải quyết thế nào để khôn
OKE Q: Em chào anh chị, cho em hỏi khi giao tiếp với thầy cô và nhà trường, m
OKE Q: Em thấy có hướng dẫn về việc chào hỏi bạn bè trong trường, vậy có cần 
OKE Q: Nếu em thấy có thông tin chưa chính thức về trường trên mạng xã hội th
OKE Q: Dạ, cho em hỏi việc giáo dục ứng xử văn minh là nhiệm vụ của riêng ai 
OKE Q: Em học ngành Kỹ thuật phần mềm, có cần chứng chỉ MOS để xét ưu tú khôn
OKE Q: Ai là người quyết định cấp Giấy chứng nhận Cử nhân/Kỹ sư ưu tú cho sin
KO Q: Các phòng ban nào trong trường có trách nhiệm triển khai quy định về x
   Gold parent: 11.2457-QD_cap_chung_nhan_ky_s_parent_0004
   Retrieved  : ['25.2020-Noi_quy_phong_thi_parent_0000', 'QĐ_số_22_vv_ban_hành_quy_định__parent_0004', '25.2020-Noi_quy_phong_thi_parent_0005', '17.Thong_tin_phong_ban_parent_0000', 'Quy_chế_Công_tác_sinh_viên_parent_0034']
KO Q: Nếu em vi

## 9. Sample Retrieval Demo

In [ ]:
sample_query = 'Điều kiện để được xét học bổng khuyến khích học tập là gì?'
results = retrieve(sample_query, k=5)

print(f'Query: {sample_query}\n')
for i, r in enumerate(results):
    print(f"[{i+1}] Score: {r['score']:.4f}  |  Source: {r['chunk']['source_file']}")
    print(f"    {r['chunk']['text'][:200]}...\n")

Query: Điều kiện để được xét học bổng khuyến khích học tập là gì?

[1] Score: 0.7438  |  Source: Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt
    III. HỌC BỔNG KHUYẾN KHÍCH HỌC TẬP
1. Đối tượng xét cấp học bổng khuyến khích học tập: Sinh viên đang học trình độ đại học tại Trường, Phân hiệu và thỏa tất cả điều kiện xét cấp học bổng khuyến khích ...

[2] Score: 0.6313  |  Source: Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt
    III. HỌC BỔNG KHUYẾN KHÍCH HỌC TẬP
1. Đối tượng xét cấp học bổng khuyến khích học tập: Sinh viên đang học trình độ đại học tại Trường, Phân hiệu và thỏa tất cả điều kiện xét cấp học bổng khuyến khích ...

[3] Score: 0.6067  |  Source: Hướng dẫn học bổng, khen thưởng, hỗ trợ người học và hỗ trợ khác.txt
    9. Học bổng khuyến khích dành cho tân sinh viên có thành tích tiếng Anh đầu vào cao
9.1. Đối tượng: Sinh viên trúng tuyển và nhập học tại Trường, Phân hiệu thuộc Trường trong đợt tuyển sinh đại học nă...

[4] Score

---
**Next step:** Run `04_sft_training.ipynb` để fine-tune LLM với QLoRA.